# IDX Stock Prediction 2026: full pipeline on Colab

Predicts the next-day return of **ASII**, **BBRI** and **TLKM** for every trading day from January to September
2026, and checks whether the predictions beat simple baselines and buy & hold.

The data covers a fixed period (1 January 2015 to 30 September 2026), so the results can be reproduced.
No GPU is needed. The whole notebook runs in about 2 minutes on a normal Colab CPU runtime.

In [ ]:
!git clone https://github.com/haikalfr04/Stocks-Prediction.git
%cd Stocks-Prediction
!pip install -q -r requirements.txt

## 1. Get the data from Yahoo Finance

The `yfinance` package downloads stock data from Yahoo Finance straight into a pandas DataFrame. Stocks listed on
the IDX use the `.JK` suffix. Change `START_DATE` and `END_DATE` below to use a different period.

In [ ]:
import pandas as pd
import yfinance as yf

START_DATE = "2015-01-01"
END_DATE = "2026-09-30"   # last day included

tickers = {"ASII": "ASII.JK", "BBRI": "BBRI.JK", "TLKM": "TLKM.JK"}

# yfinance treats `end` as exclusive, so add one day to include END_DATE itself.
end_exclusive = (pd.Timestamp(END_DATE) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

data = {}
for code, ticker in tickers.items():
    # auto_adjust=False keeps both Close (the price shown on the IDX) and Adj Close
    # (adjusted for dividends and stock splits).
    df = yf.download(ticker, start=START_DATE, end=end_exclusive, auto_adjust=False, progress=False)
    if isinstance(df.columns, pd.MultiIndex):  # newer yfinance versions add a ticker level
        df.columns = df.columns.get_level_values(0)
    data[code] = df
    print(f"{code}: {len(df):,} trading days, {df.index[0]:%Y-%m-%d} to {df.index[-1]:%Y-%m-%d}")

The last 5 trading days of each stock (Open, High, Low, Close, Adj Close, Volume):

In [ ]:
for code, df in data.items():
    print(code)
    display(df.tail())

Closing prices of the three stocks side by side, and the summary statistics for 2026:

In [ ]:
close = pd.DataFrame({code: df["Close"] for code, df in data.items()})
display(close.tail(10))
display(close.loc["2026"].describe().round(0))

In [ ]:
ax = close.plot(subplots=True, figsize=(10, 7), title="Closing price (Rp)", legend=True)

Return in 2026, and the daily return statistics over the whole period:

In [ ]:
last_2025 = close.loc[:"2025"].iloc[-1]
latest = close.iloc[-1]
daily_return = close.pct_change()

summary = pd.DataFrame({
    "Close end of 2025": last_2025,
    f"Close {close.index[-1]:%Y-%m-%d}": latest,
    "Return 2026": (latest / last_2025 - 1).map("{:+.1%}".format),
    "Average daily return": daily_return.mean().map("{:+.3%}".format),
    "Daily volatility": daily_return.std().map("{:.2%}".format),
    "Days with no change": (daily_return == 0).mean().map("{:.1%}".format),
})
display(summary)

Company information and recent dividends. Company information is the current value from Yahoo Finance, not the
value on `END_DATE`. Some fields can be empty when Yahoo Finance has no value for them.

In [ ]:
rows = []
for code, ticker in tickers.items():
    info = yf.Ticker(ticker).info
    rows.append({
        "Code": code,
        "Name": info.get("longName"),
        "Sector": info.get("sector"),
        "Industry": info.get("industry"),
        "Market cap (Rp)": info.get("marketCap"),
        "P/E": info.get("trailingPE"),
    })
display(pd.DataFrame(rows).set_index("Code"))

In [ ]:
for code, ticker in tickers.items():
    dividends = yf.Ticker(ticker).dividends
    print(code, "dividends per share (Rp):")
    display(dividends[dividends.index.tz_localize(None) <= END_DATE].tail(5))

Save the data to `data/idx_prices.xlsx`, with one sheet per stock and an `Info` sheet. All later steps read this
file, so the data is downloaded only once.

In [ ]:
from src.download import save_excel
save_excel(data, "data/idx_prices.xlsx", START_DATE, END_DATE)

Optional: download the Excel file to open it on your computer.

In [ ]:
from google.colab import files
files.download("data/idx_prices.xlsx")

## 2. Exploratory data analysis

Summary statistics, price history, the distribution of daily returns, and their autocorrelation. Autocorrelation
close to zero means past returns say little about the next return, which already hints at how hard the task is.

In [ ]:
!python -m src.eda
from IPython.display import Image, Markdown, display
display(Markdown(open("results/eda_stats.md").read()))
for name in ["price_history", "return_distribution", "autocorrelation"]:
    display(Image(f"results/figures/{name}.png", width=800))

## 3. Walk-forward prediction for 2026

Models are trained only on data before the day they predict. The first model is trained on 2015-2025 data, predicts
the next 21 trading days, and is then retrained with those days added. This repeats until the end of the data.

Five models are compared: three baselines (random walk, historical mean, momentum), Ridge regression and LightGBM.
Each prediction is also used in a long-or-cash trading strategy with IDX fees, compared with buy & hold.

In [ ]:
!python -m src.run

In [ ]:
display(Markdown(open("results/comparison.md").read()))

In [ ]:
for name in ["model_comparison", "ASII_2026_predictions", "BBRI_2026_predictions", "TLKM_2026_predictions",
             "equity_curves", "monthly_accuracy", "feature_importance"]:
    display(Image(f"results/figures/{name}.png", width=800))

Daily predictions for 2026 (one row per trading day) are saved in `results/predictions/`.

In [ ]:
daily = pd.read_csv("results/predictions/BBRI_2026.csv", index_col=0, parse_dates=True)
daily[["prev_close", "actual_close", "pred_close_lightgbm", "actual_return", "pred_return_lightgbm"]].tail(10)

## 4. Prediction for the day after the data ends

LightGBM trained on all the data predicts the first trading day after `END_DATE`.
For learning purposes only. This is not investment advice.

In [ ]:
!python -m src.predict

## 5. Save the results to GitHub

Download `results.zip`, unzip it into the repository and commit the `results/` and `data/` folders. Then save this
notebook with its outputs (**File → Save a copy in GitHub**), so visitors can see every table and chart on GitHub
without running anything.

In [ ]:
!zip -qr results.zip results data/idx_prices.xlsx
from google.colab import files
files.download("results.zip")